# 机器学习基线对照实验（ML_Funning）

使用与 LoRA 实验一致的数据划分与超参数，输出统一指标格式。

In [1]:
"""
第一部分：导入依赖
"""

import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data

print("✅ 所有库导入完成")

C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [2]:
"""
第二部分：统一配置
"""

MODEL_NAME = "LogisticRegression-ML"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]

MAX_FEATURES = CONFIG["CLASSIC_MAX_FEATURES"]
NGRAM_RANGE = CONFIG["CLASSIC_NGRAM_RANGE"]
MIN_DF = CONFIG["CLASSIC_MIN_DF"]
MAX_DF = CONFIG["CLASSIC_MAX_DF"]

EXP_NAME = "机器学习基线对照实验"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"模型名称: {MODEL_NAME}")
print(f"数据目录: {DATA_PATH}")
print(f"随机种子: {RANDOM_SEED}")
print(f"最大特征数: {MAX_FEATURES}")
print(f"N-gram范围: {NGRAM_RANGE}")
print(f"最小文档频率: {MIN_DF}")
print(f"最大文档频率: {MAX_DF}")
print("=" * 50)

✅ 配置加载完成


In [3]:
"""
第三部分：定义工具函数
"""


def build_vectorizer():
    """
    目的：构建TF-IDF向量化器，用于将文本转换为数值特征向量
    
    数据流向：
    输入：无（使用全局配置参数）
    输出：TfidfVectorizer对象（已配置好的向量化器）
    
    操作过程：
    1. 创建TfidfVectorizer对象，设置以下参数：
       - analyzer="char": 按字符级别分析（适合中文文本）
       - ngram_range: N-gram范围，例如(1,3)表示提取1-gram、2-gram、3-gram特征
       - max_features: 保留的最大特征数量，控制特征维度
       - min_df: 最小文档频率，过滤掉出现次数太少的特征
       - max_df: 最大文档频率，过滤掉出现次数太多的特征（如停用词）
    2. 返回配置好的向量化器对象
    
    应用场景：
    在训练前调用，用于将原始文本转换为机器学习模型可以处理的数值特征
    """
    return TfidfVectorizer(
        analyzer="char",
        ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
        min_df=MIN_DF,
        max_df=MAX_DF,
    )


def evaluate(y_true, y_pred, y_prob=None):
    """
    目的：计算模型在数据集上的多项评估指标
    
    数据流向：
    输入：
      - y_true: 真实标签（数组或列表）
      - y_pred: 预测标签（数组或列表）
      - y_prob: 预测概率（可选，用于计算损失）
    输出：
      - (acc, precision, recall, f1, loss) 五元组
    
    操作过程：
    1. 使用sklearn计算准确率（accuracy）：正确预测的样本比例
    2. 使用sklearn计算精确率、召回率、F1分数：
       - precision: 预测为正的样本中真正为正的比例
       - recall: 真正为正的样本中被正确预测的比例
       - f1: 精确率和召回率的调和平均数
       - average="binary": 针对二分类任务
       - zero_division=0: 避免除零错误
    3. 如果提供了预测概率，使用log_loss计算交叉熵损失
    4. 返回所有指标的元组
    
    应用场景：
    在训练集、验证集、测试集上评估模型性能，获得全面的评估指标
    """
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    loss = None
    if y_prob is not None:
        loss = log_loss(y_true, y_prob)
    return acc, precision, recall, f1, loss


def print_report(train_metrics, val_metrics, test_metrics, elapsed_sec):
    """
    目的：格式化打印训练、验证、测试集的评估结果
    
    数据流向：
    输入：
      - train_metrics: 训练集指标元组 (acc, precision, recall, f1, loss)
      - val_metrics: 验证集指标元组
      - test_metrics: 测试集指标元组
      - elapsed_sec: 训练耗时（秒）
    输出：无（直接打印到控制台）
    
    操作过程：
    1. 将训练时间从秒转换为分钟
    2. 打印分隔线和模型名称
    3. 依次打印训练集、验证集、测试集的五项指标：
       - Acc: 准确率
       - Precision: 精确率
       - Recall: 召回率
       - F1: F1分数
       - Loss: 交叉熵损失（如果为None则显示N/A）
    4. 打印训练时间（秒和分钟）
    5. 打印结束分隔线
    
    应用场景：
    实验结束后，以统一格式展示所有数据集上的评估结果，便于对比不同模型
    """
    elapsed_min = elapsed_sec / 60
    
    # 辅助函数：格式化loss值，处理None情况
    def format_loss(loss):
        return f"{loss:.3f}" if loss is not None else "N/A"
    
    print("\n" + "=" * 50)
    print(f"模型: {MODEL_NAME}")
    print(
        f"训练集 - Acc: {train_metrics[0]:.3f} | Precision: {train_metrics[1]:.3f} | "
        f"Recall: {train_metrics[2]:.3f} | F1: {train_metrics[3]:.3f} | "
        f"Loss: {format_loss(train_metrics[4])}"
    )
    print(
        f"验证集 - Acc: {val_metrics[0]:.3f} | Precision: {val_metrics[1]:.3f} | "
        f"Recall: {val_metrics[2]:.3f} | F1: {val_metrics[3]:.3f} | "
        f"Loss: {format_loss(val_metrics[4])}"
    )
    print(
        f"测试集 - Acc: {test_metrics[0]:.3f} | Precision: {test_metrics[1]:.3f} | "
        f"Recall: {test_metrics[2]:.3f} | F1: {test_metrics[3]:.3f} | "
        f"Loss: {format_loss(test_metrics[4])}"
    )
    print(f"训练时间: {elapsed_sec:.1f} 秒 ({elapsed_min:.2f} 分钟)")
    print("=" * 50)

print("✅ 工具函数定义完成")

✅ 工具函数定义完成


In [4]:

"""
第四部分：准备数据
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vectorizer = build_vectorizer()
x_train = vectorizer.fit_transform(train_df["review"])
x_val = vectorizer.transform(val_df["review"])
x_test = vectorizer.transform(test_df["review"])

y_train = train_df["label"].astype(int).values
y_val = val_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

print("✅ 数据准备完成")



✅ 数据准备完成


In [ ]:
"""
第五部分：训练与评估
"""

model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)

start = time.time()
model.fit(x_train, y_train)
elapsed = time.time() - start

train_pred = model.predict(x_train)
val_pred = model.predict(x_val)
test_pred = model.predict(x_test)

train_prob = model.predict_proba(x_train)
val_prob = model.predict_proba(x_val)
test_prob = model.predict_proba(x_test)

train_metrics = evaluate(y_train, train_pred, train_prob)
val_metrics = evaluate(y_val, val_pred, val_prob)
test_metrics = evaluate(y_test, test_pred, test_prob)
print("\n实验名称:"+EXP_NAME)
print_report(train_metrics, val_metrics, test_metrics, elapsed)
